Till now :

##### Hive Style Partitioning:
- we have hive style partitioning where partition value gets its own folder. 
- All of the records for that partition value go to that folder.

##### Zorder
- colocate similar data in one file

##### Problems :
- But these are not flexible meaning you have to decide your partitioning column or zorder column upfront.
- If today maximum query column changes from country to category we have to re-write everything which is a big problem.


##### Liquid Clustering
- It allows you to **`change the clustering columns anytime`**.
- Algorithm underneath makes sure :
     - Uniform File Size
     - Appropriate Number of Files
     - Data is CoLocated, Similar data in same file.

In [0]:
%fs ls abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/

In [0]:
df1 = spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_101_200.parquet")
df2 = spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_1_100.parquet")
df3 = spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_201_99457.parquet")

In [0]:
df_union = df1.union(df2).union(df3)
df_union = df_union.select("customer_id","category","price","quantity","invoice_date")

In [0]:
df_union.write.mode("overwrite").partitionBy("invoice_date").saveAsTable("deltacatalog.deltadb.lc_ex1")

In [0]:
%sql
OPTIMIZE deltacatalog.deltadb.lc_ex1
ZORDER BY customer_id

In [0]:
%sql
Select count(*) from deltacatalog.deltadb.lc_ex1;

In [0]:
df_union.write.mode("overwrite").clusterBy("invoice_date","customer_id").saveAsTable("deltacatalog.deltadb.lc_ex2")

In [0]:
%sql
Select count(*) from deltacatalog.deltadb.lc_ex2;

In [0]:
%%time
spark.sql("""
          
    select category, 
    sum(price*quantity) as total_sales
    where customer_id = 201 and (invoice_date between '2022-01-01' and '2023-12-31')
    from deltacatalog.deltadb.lc_ex1
    group by category

""").display()

In [0]:
%%time
spark.sql("""
          
    select category, 
    sum(price*quantity) as total_sales
    where customer_id = 201 and (invoice_date between '2022-01-01' and '2023-12-31')
    from deltacatalog.deltadb.lc_ex2
    group by category

""").display()